# Lab 02 · Reference solution

The polished final implementation of [Lab 02: Tool design and selection](../README.md).

Ships only `tools_v1` (the working design) — the lab notebook walks
through `tools_v0` and `tools_v1` side by side to demonstrate *why*
the v1 shape works. The lab is where the diagnosis lives; this solution
is the final state.

> ⏱ Read time: ~6 min · Notebook ~18 cells.
> 📖 The lab's "Step 4: Diagnose" section explains why `tools_v0`'s
> generic `update_customer`/`update_order` failed: ambiguous selection
> between similar tools, no confirmation gate on destructive operations,
> permissive `status` strings instead of enum, terse descriptions. The
> v1 design below fixes each.

## Setup

In [ ]:
import json
import os
import pathlib
from dataclasses import dataclass, field
from typing import Any, Literal

from dotenv import load_dotenv
from pydantic import BaseModel, ConfigDict, Field

here = pathlib.Path.cwd()
for parent in [here, *here.parents]:
    if (parent / ".env.example").exists():
        load_dotenv(parent / ".env")
        break

assert os.getenv("OPENAI_API_KEY") or os.getenv("ANTHROPIC_API_KEY")

PROVIDER = "openai"
MODEL = {
    "openai": "gpt-4o-mini",
    "anthropic": "claude-haiku-4-5-20251001",
}[PROVIDER]
print(f"Using {PROVIDER} / {MODEL}")


## Mock backend

Three canned tables: `CUSTOMERS`, `ORDERS`, `INVENTORY`. Real backends
would be a database; the tool *shape* is what we're studying.

In [ ]:
CUSTOMERS = {
    1001: {"id": 1001, "email": "ada@example.com", "name": "Ada Lovelace",
           "plan": "pro", "status": "active"},
    1002: {"id": 1002, "email": "alan@example.com", "name": "Alan Turing",
           "plan": "free", "status": "active"},
    1003: {"id": 1003, "email": "grace@example.com", "name": "Grace Hopper",
           "plan": "pro", "status": "suspended"},
}

ORDERS = {
    9001: {"id": 9001, "customer_id": 1001, "total": 42.50, "status": "shipped"},
    9002: {"id": 9002, "customer_id": 1001, "total": 19.99, "status": "pending"},
    9003: {"id": 9003, "customer_id": 1003, "total": 99.00, "status": "shipped"},
}

INVENTORY = {
    "SKU-A": {"sku": "SKU-A", "name": "Cable", "stock": 47, "price": 9.99},
    "SKU-B": {"sku": "SKU-B", "name": "Adapter", "stock": 0, "price": 19.99},
    "SKU-C": {"sku": "SKU-C", "name": "Hub", "stock": 12, "price": 39.99},
}


def _find_by_email(email: str) -> dict | None:
    for cust in CUSTOMERS.values():
        if cust["email"].lower() == email.lower():
            return cust
    return None


## Provider-agnostic chat client

Same shape as Lab 01, plus `parallel_tool_calls` and `tool_choice`
support since both matter for tool routing.

In [ ]:
@dataclass
class ToolCall:
    id: str
    name: str
    arguments: dict


@dataclass
class AssistantMessage:
    content: str | None
    tool_calls: list[ToolCall] = field(default_factory=list)


def chat_with_tools(
    messages: list[dict],
    tools: list[dict] | None = None,
    tool_choice: str = "auto",
    parallel_tool_calls: bool = True,
) -> AssistantMessage:
    if PROVIDER == "openai":
        from openai import OpenAI
        resp = OpenAI().chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=tools,
            tool_choice=tool_choice if tools else None,
            parallel_tool_calls=parallel_tool_calls if tools else None,
            temperature=0,
        )
        msg = resp.choices[0].message
        return AssistantMessage(
            content=msg.content,
            tool_calls=[
                ToolCall(
                    id=tc.id, name=tc.function.name,
                    arguments=json.loads(tc.function.arguments),
                )
                for tc in (msg.tool_calls or [])
            ],
        )
    elif PROVIDER == "anthropic":
        from anthropic import Anthropic
        client = Anthropic()
        system = next((m["content"] for m in messages if m["role"] == "system"), "")
        non_system = [m for m in messages if m["role"] != "system"]
        anth_tools = [
            {"name": t["function"]["name"],
             "description": t["function"]["description"],
             "input_schema": t["function"]["parameters"]}
            for t in (tools or [])
        ]
        resp = client.messages.create(
            model=MODEL, system=system, messages=non_system,
            tools=anth_tools or None, max_tokens=1024, temperature=0,
        )
        text, tcs = "", []
        for block in resp.content:
            if block.type == "text":
                text += block.text
            elif block.type == "tool_use":
                tcs.append(ToolCall(id=block.id, name=block.name, arguments=dict(block.input)))
        return AssistantMessage(content=text or None, tool_calls=tcs)
    raise ValueError(f"Unknown PROVIDER: {PROVIDER!r}")


## `tools_v1` — the working design

The four design principles from
[`concepts/tools/tool-design.md`](../../../concepts/tools/tool-design.md),
applied:

1. **One tool per intent.** `lookup_customer_by_email` and
   `lookup_customer_by_id` are separate; the model picks by name
   instead of a fuzzy `mode` argument.
2. **`Literal` enum for status.** `OrderStatus` constrains the schema;
   the model can't invent invalid statuses.
3. **`confirmed: bool` on destructive transitions.** Cancelling an
   order requires `confirmed=true`; without it, the tool returns
   `confirmation_required` and the agent asks the user.
4. **Descriptions with negative guidance.** "Do NOT use this for X — use
   Y instead" — the single most reliable fix for selection drift
   between overlapping tools.

`StrictModel` sets `extra="forbid"` so the model can't pass extra fields
under OpenAI's strict function-calling mode.

In [ ]:
class StrictModel(BaseModel):
    model_config = ConfigDict(extra="forbid")


# ── Customer lookup (split by intent) ───────────────────────────────────

class LookupByEmailArgs(StrictModel):
    email: str = Field(description="Customer's exact email address (case-insensitive).")


def lookup_customer_by_email(args: LookupByEmailArgs) -> dict:
    cust = _find_by_email(args.email)
    if cust is None:
        return {"error": "not_found", "email": args.email}
    return {"customer": cust}


class LookupByIdArgs(StrictModel):
    customer_id: int = Field(description="Internal customer id (integer, e.g. 1001).")


def lookup_customer_by_id(args: LookupByIdArgs) -> dict:
    cust = CUSTOMERS.get(args.customer_id)
    if cust is None:
        return {"error": "not_found", "customer_id": args.customer_id}
    return {"customer": cust}


# ── Order reads ──────────────────────────────────────────────────────────

class ListOrdersArgs(StrictModel):
    customer_id: int = Field(description="Customer id whose orders to list.")


def list_orders_for_customer(args: ListOrdersArgs) -> dict:
    rows = [o for o in ORDERS.values() if o["customer_id"] == args.customer_id]
    return {
        "orders": [{"id": o["id"], "total": o["total"], "status": o["status"]} for o in rows],
        "count": len(rows),
    }


class GetOrderArgs(StrictModel):
    order_id: int = Field(description="Order id to fetch.")


def get_order(args: GetOrderArgs) -> dict:
    order = ORDERS.get(args.order_id)
    if order is None:
        return {"error": "not_found", "order_id": args.order_id}
    return {"order": dict(order)}


# ── Order update with confirmation gate ──────────────────────────────────

OrderStatus = Literal["pending", "shipped", "delivered", "cancelled"]


class UpdateOrderArgs(StrictModel):
    order_id: int = Field(description="Order id to update.")
    new_status: OrderStatus = Field(
        description="New status. Must be one of: pending, shipped, delivered, cancelled."
    )
    confirmed: bool = Field(
        description=(
            "Must be true to apply destructive transitions (cancellations). "
            "Set to true ONLY after the user has explicitly confirmed."
        )
    )


def update_order(args: UpdateOrderArgs) -> dict:
    order = ORDERS.get(args.order_id)
    if order is None:
        return {"error": "not_found", "order_id": args.order_id}
    if args.new_status == "cancelled" and not args.confirmed:
        return {
            "error": "confirmation_required",
            "message": (
                "Cancelling an order is destructive. Ask the user to confirm, "
                "then call again with confirmed=true."
            ),
            "current_status": order["status"],
        }
    old = order["status"]
    order["status"] = args.new_status
    return {"ok": True, "order_id": args.order_id, "old_status": old, "new_status": args.new_status}


# ── Inventory ────────────────────────────────────────────────────────────

class CheckStockArgs(StrictModel):
    sku: str = Field(description="SKU code, e.g. 'SKU-A'.")


def check_stock(args: CheckStockArgs) -> dict:
    item = INVENTORY.get(args.sku)
    if item is None:
        return {"error": "not_found", "sku": args.sku}
    return {"sku": item["sku"], "name": item["name"], "stock": item["stock"], "price": item["price"]}


# ── Registry: name → (handler, args_model, description-with-negative-guidance) ──

TOOLS_V1 = {
    "lookup_customer_by_email": (
        lookup_customer_by_email, LookupByEmailArgs,
        "Look up a customer by exact email address. Use this when you have an email "
        "and need profile info. Do NOT use this for fuzzy or partial-match search. "
        "Returns the customer object, or error 'not_found' if no match.",
    ),
    "lookup_customer_by_id": (
        lookup_customer_by_id, LookupByIdArgs,
        "Look up a customer by their internal numeric id. Do NOT use this when you "
        "only have an email — use lookup_customer_by_email instead.",
    ),
    "list_orders_for_customer": (
        list_orders_for_customer, ListOrdersArgs,
        "List all orders for a given customer id. Returns a list of orders with id, "
        "total, and status. Do NOT use this to fetch one order; use get_order.",
    ),
    "get_order": (
        get_order, GetOrderArgs,
        "Fetch a single order by id. Returns the full order record.",
    ),
    "update_order": (
        update_order, UpdateOrderArgs,
        "Update an order's status. For destructive transitions (cancellations), the "
        "confirmed flag MUST be true and the user MUST have explicitly confirmed. "
        "Returns 'confirmation_required' error if confirmed=false on a destructive change.",
    ),
    "check_stock": (
        check_stock, CheckStockArgs,
        "Check inventory stock for a specific SKU code. Returns name, stock, and price.",
    ),
}


## Schemas + dispatcher + loop

In [ ]:
def make_schemas(tools: dict) -> list[dict]:
    """OpenAI-format tool schemas built from a `{name: (fn, args_model, description)}` registry."""
    return [
        {
            "type": "function",
            "function": {
                "name": name,
                "description": description,
                "parameters": args_model.model_json_schema(),
            },
        }
        for name, (_fn, args_model, description) in tools.items()
    ]


def execute_tool(call: ToolCall, tools: dict) -> dict:
    """Dispatch a tool call; return structured result/error."""
    if call.name not in tools:
        return {"error": "unknown_tool", "tool": call.name, "available": list(tools)}
    fn, args_model, _ = tools[call.name]
    try:
        return fn(args_model.model_validate(call.arguments))
    except Exception as e:
        return {"error": "exception", "type": type(e).__name__, "detail": str(e)}


MAX_STEPS = 8


def run_agent(question: str, tools: dict, *, tool_choice: str = "auto", verbose: bool = True) -> dict:
    """Run the agent against `tools` until final answer or step cap."""
    system_prompt = (
        "You are a customer-support assistant. For each step, briefly state your "
        "reasoning, then call the most appropriate tool. When you have enough "
        "information, give a final answer without calling a tool. For destructive "
        "actions (cancellations), always ask the user to confirm before calling "
        "update_order with confirmed=true."
    )
    state: list[dict] = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": question},
    ]
    schemas = make_schemas(tools)
    trace: list[dict] = []

    for step in range(MAX_STEPS):
        msg = chat_with_tools(state, tools=schemas, tool_choice=tool_choice)
        state.append({
            "role": "assistant",
            "content": msg.content,
            "tool_calls": [
                {"id": tc.id, "type": "function",
                 "function": {"name": tc.name, "arguments": json.dumps(tc.arguments)}}
                for tc in msg.tool_calls
            ] if msg.tool_calls else None,
        })
        if not msg.tool_calls:
            trace.append({"step": step + 1, "final": msg.content})
            if verbose:
                print(f"── Step {step + 1}: FINAL: {msg.content}")
            return {"final": msg.content, "steps": step + 1, "trace": trace}
        for call in msg.tool_calls:
            result = execute_tool(call, tools)
            trace.append({"step": step + 1, "tool": call.name, "args": call.arguments, "result": result})
            if verbose:
                print(f"── Step {step + 1}: {call.name}({call.arguments}) → {result}")
            state.append({
                "role": "tool",
                "tool_call_id": call.id,
                "content": json.dumps(result),
            })
    return {"final": "[step cap reached]", "steps": MAX_STEPS, "trace": trace}


## Demo: routine query

Customer lookup → list orders. Two-step, no confirmation needed.

In [ ]:
_ = run_agent("Show me the orders for ada@example.com", TOOLS_V1)


**Sample output (will vary slightly):**

```
── Step 1: lookup_customer_by_email({'email': 'ada@example.com'}) → {'customer': {'id': 1001, ...}}
── Step 2: list_orders_for_customer({'customer_id': 1001}) → {'orders': [...], 'count': 2}
── Step 3: FINAL: Ada (id 1001) has 2 orders: #9001 ($42.50, shipped) and #9002 ($19.99, pending).
```

## Demo: the confirmation gate

A destructive request. The first attempt to cancel should bounce off
the `confirmation_required` gate; the agent should surface the
confirmation to the user rather than just retrying with
`confirmed=true`.

In [ ]:
_ = run_agent("Cancel order 9002 for me", TOOLS_V1)


**Sample output:**

```
── Step 1: update_order({'order_id': 9002, 'new_status': 'cancelled', 'confirmed': false})
          → {'error': 'confirmation_required', 'message': '...', 'current_status': 'pending'}
── Step 2: FINAL: Order 9002 is currently pending ($19.99). Cancelling is permanent.
          Confirm you want to cancel and I'll proceed.
```

The agent correctly bounces off the gate and asks the user. Compare to
`tools_v0` (in the lab) which cancelled immediately without asking — the
safety lives in the *schema and return contract*, not in the prompt.

## `tool_choice` demonstration

Three values worth knowing:

- `auto` — model decides (default).
- `none` — model must respond without calling a tool. Useful for the
  *synthesis* step of a multi-step trajectory.
- A specific tool name — model must call that tool. Useful for
  forced-routing.

In [ ]:
# Force a no-tool response — useful for the synthesis step
no_tool_response = chat_with_tools(
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "What is ReAct, in one sentence?"},
    ],
    tools=make_schemas(TOOLS_V1),
    tool_choice="none",
)
print(f"Content: {no_tool_response.content}")
print(f"Tool calls: {no_tool_response.tool_calls}")


**Sample output:**

```
Content: ReAct (Reason + Act) is a prompting pattern where an LLM alternates between explicit reasoning steps and tool calls, with each next decision conditioned on the most recent observation.
Tool calls: []
```

## Production readiness — out of scope here

For a real deployment add: idempotency keys on destructive operations
(so a retried cancel doesn't double-cancel), audit logging of every
destructive call, per-tool authorization checks (does the user own this
order?), and rate-limiting on destructive tools. The tool *design*
above is the foundation — those wrap the handlers.